<a href="https://colab.research.google.com/github/TerteryanTatev/Optimization-Methods/blob/main/Linear_Regression_Feature_Selection/regression_feature_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import combinations

In [ ]:
df = pd.DataFrame({
    'N': [ i for i in range(1,16)],
    'X1': [1, 1, 2, 1, 1, 2, 2, 2, 1, 1, 2, 1, 2, 2, 2],
    'X2': [4, 3, 1, 2, 2, 1, 4, 3, 2, 1, 2, 3, 4, 4, 1],
    'X3': [2, 1, 5, 1, 4, 3, 2, 5, 3, 1, 4, 3, 5, 1, 3],
    'X4': [2, 2, 3, 2, 5, 4, 3, 4, 1, 2, 2, 5, 4, 1, 2],
    'Y':  [2, 2, 3, 2, 3, 3, 3, 4, 2, 1, 2, 3, 4, 3, 3]
})

df

,N,X1,X2,X3,X4,Y
0,1,1,4,2,2,2
1,2,1,3,1,2,2
2,3,2,1,5,3,3
3,4,1,2,1,2,2
4,5,1,2,4,5,3
5,6,2,1,3,4,3
6,7,2,4,2,3,3
7,8,2,3,5,4,4
8,9,1,2,3,1,2
9,10,1,1,1,2,1


In [ ]:
A = np.array([
    [20, df['X1'].sum(),df['X2'].sum(), df['X3'].sum(), df['X4'].sum()],
    [df['X1'].sum(), df['X1'] @ df['X1'], df['X1'] @ df['X2'], df['X1'] @ df['X3'], df['X1'] @ df['X4']],
    [df['X2'].sum(), df['X1'] @ df['X2'], df['X2'] @ df['X2'], df['X2'] @ df['X3'], df['X2'] @ df['X4']],
    [df['X3'].sum(), df['X1'] @ df['X3'], df['X3'] @ df['X2'], df['X3'] @ df['X3'], df['X3'] @ df['X4']],
    [df['X4'].sum(), df['X1'] @ df['X4'], df['X4'] @ df['X2'], df['X4'] @ df['X3'], df['X4'] @ df['X4']]

])
A

array([[ 20,  23,  37,  43,  42],
       [ 23,  39,  57,  71,  65],
       [ 37,  57, 111, 103, 104],
       [ 43,  71, 103, 155, 136],
       [ 42,  65, 104, 136, 142]])

In [ ]:
C = np.array([
    df['Y'].sum(),
    df['X1'] @ df['Y'],
    df['X2'] @ df['Y'],
    df['X3'] @ df['Y'],
    df['X4'] @ df['Y']
])
C

array([ 40,  65, 103, 126, 121])

In [ ]:
A_inv = np.linalg.inv(A)
b = A_inv @ C
# det = np.linalg.det(A)
# det
b


array([-0.01988269,  0.70646692,  0.21301692,  0.13196577,  0.25220858])

In [ ]:
df['Y_hat'] = [b[0]+ df.iloc[i].to_numpy()[1:5] @ b[1:] for i in range(15)]
df


,N,X1,X2,X3,X4,Y,Y_hat
0,1,1,4,2,2,2,2.307001
1,2,1,3,1,2,2,1.962018
2,3,2,1,5,3,3,3.022523
3,4,1,2,1,2,2,1.749001
4,5,1,2,4,5,3,2.901524
5,6,2,1,3,4,3,3.010800
6,7,2,4,2,3,3,3.265676
7,8,2,3,5,4,4,3.700765
8,9,1,2,3,1,2,1.760724
9,10,1,1,1,2,1,1.535984


In [ ]:
mse = ((df["Y"] - df["Y_hat"]) ** 2).mean()
mse

np.float64(0.11915419219745281)

In [ ]:

Y_mean = np.mean(df["Y"])
SSO = np.sum((df["Y"] - Y_mean) ** 2)
SSR = np.sum((df["Y_hat"] - Y_mean) ** 2)
SSE = np.sum((df["Y"] - df["Y_hat"]) ** 2)
SSO, SSR, SSE


(np.float64(9.333333333333332),
 np.float64(7.011862198715259),
 np.float64(1.7873128829617921))

In [ ]:
R_hat = SSR / SSO
R_hat

np.float64(0.7512709498623493)

In [ ]:
msr = SSR / 4
msr

np.float64(1.7529655496788148)

In [ ]:
sigma = SSE / (15-4-1)
sigma

np.float64(0.1787312882961792)

In [ ]:
f = msr / sigma
f

np.float64(9.807826969690614)

In [ ]:
n = 15

def calc_R2(y, X):
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    return model.rsquared

R_full = calc_R2(df['Y'], df[['X1', 'X2', 'X3', 'X4']])
R_full

np.float64(0.8098208236724835)

In [ ]:
variables = ['X1', 'X2', 'X3', 'X4']
results = {}

for combo in combinations(variables, 3):
    X = df[list(combo)]
    R2 = calc_R2(df['Y'], X)
    results["R(" + "".join(combo) + ")"] = R2
results

{'R(X1X2X3)': np.float64(0.6963805683563749),
 'R(X1X2X4)': np.float64(0.7824269843636829),
 'R(X1X3X4)': np.float64(0.7080917970300962),
 'R(X2X3X4)': np.float64(0.6504263389357205)}

In [ ]:

best_model = max(results, key=results.get)
R2_best = results[best_model]
R2_best

np.float64(0.7824269843636829)

In [ ]:

F = ((R_full - R2_best) * (15-4-1) )/ (1- R_full )
F

np.float64(1.4404226497239883)

In [ ]:
variables = ['X1', 'X2', 'X4']
results = {}

for combo in combinations(variables, 2):
    X = df[list(combo)]
    R1 = calc_R2(df['Y'], X)
    results["R(" + "".join(combo) + ")"] = R1
results

{'R(X1X2)': np.float64(0.4759316770186337),
 'R(X1X4)': np.float64(0.697701270074525),
 'R(X2X4)': np.float64(0.4508409180846854)}

In [ ]:

best_model = max(results, key=results.get)
R1_best = results[best_model]
R1_best

np.float64(0.697701270074525)

In [ ]:

F1 = ((R2_best - R1_best) * (15-3-1) )/ (1- R2_best )

F1

np.float64(4.283540651652257)

In [ ]:
variables = ['X1', 'X4']
results = {}

for combo in combinations(variables, 1):
    X = df[list(combo)]
    R2 = calc_R2(df['Y'], X)
    results["R(" + "".join(combo) + ")"] = R2
results

{'R(X1)': np.float64(0.38584183673469385),
 'R(X4)': np.float64(0.35567915690866503)}

In [ ]:

best_model = max(results, key=results.get)
R0_best = results[best_model]
R0_best

np.float64(0.38584183673469385)

In [ ]:

F2 = ((R1_best - R0_best) * (15-3-1) )/ (1- R1_best )

F2

np.float64(11.347893415178568)